# 05 · Window functions (Oracle)

A **window function** computes a value from a group of rows (the *window*) but **keeps every row**
(GROUP BY would squash them into one).

```
function() OVER (PARTITION BY <groups> ORDER BY <sort> ROWS BETWEEN ... )
```
- `PARTITION BY` = restart for each group (like GROUP BY, but rows stay)
- `ORDER BY` = the order inside the group (needed for ranking, running totals, LAG/LEAD)
- frame (`ROWS BETWEEN ...`) = which rows around the current row to use

You **cannot** use a window function in WHERE. Wrap it in a subquery/CTE and filter outside.
BigQuery (and Oracle 23ai) also has `QUALIFY`. Spark SQL has no QUALIFY.

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/05_window_functions.ipynb`](../../oracle/practice/05_window_functions.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. ROW_NUMBER vs RANK vs DENSE_RANK  ·  🟢 easy

For department 2 (Sales), show `name`, `salary` and three columns ordered by salary (high first):
`rn` (ROW_NUMBER, ties broken by name), `rnk` (RANK), `drnk` (DENSE_RANK). Look at the 90,000 tie: what does each function do?

*Expected result: 6 rows · columns: `name`, `salary`, `rn`, `rnk`, `drnk`*

**How it works:** All three number rows by salary (high first) inside one window. They differ only on ties: ROW_NUMBER never repeats, RANK repeats then skips, DENSE_RANK repeats without gaps.

In [ ]:
%%oracle
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS rn,
       RANK()       OVER (ORDER BY salary DESC)       AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC)       AS drnk
FROM employees
WHERE dept_id = 2
ORDER BY rn

,name,salary,rn,rnk,drnk
0,Henry Clark,140000.0,1,1,1
1,Irene Lewis,90000.0,2,2,2
2,Jack Walker,90000.0,3,2,2
3,Karen Hall,90000.0,4,2,2
4,Leo Allen,70000.0,5,5,3
5,Mia Young,65000.0,6,6,4


**Why / traps:** Ties 90k ×3: ROW_NUMBER → 2,3,4 (always different). RANK → 2,2,2 then **5** (skips). DENSE_RANK → 2,2,2 then **3** (no gaps). Most asked window question.

## Q2. Top 2 salary levels per department  ·  🟡 medium

For each department (skip NULL dept_id) show employees whose salary is in the **top 2 different salary values** of that department. Show `dept_id`, `name`, `salary`.

*Expected result: 15 rows · columns: `dept_id`, `name`, `salary`*

**How it works:** 1) DENSE_RANK salaries inside each department (`PARTITION BY dept_id`). 2) Keep rank ≤ 2 in an outer query (or with QUALIFY in BigQuery).

In [ ]:
%%oracle
SELECT dept_id, name, salary
FROM (SELECT dept_id, name, salary,
             DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS drnk
      FROM employees
      WHERE dept_id IS NOT NULL) t
WHERE drnk <= 2
ORDER BY dept_id, salary DESC, name

,dept_id,name,salary
0,1,Alice Johnson,250000.0
1,1,Bob Smith,150000.0
2,2,Henry Clark,140000.0
3,2,Irene Lewis,90000.0
4,2,Jack Walker,90000.0
5,2,Karen Hall,90000.0
6,3,Nathan King,130000.0
7,3,Olivia Scott,85000.0
8,3,Paul Adams,85000.0
9,4,Rachel Nelson,135000.0


**Why / traps:** "Top N **people**" → ROW_NUMBER. "Top N **salaries**, keep ties" → DENSE_RANK. Ask which one!

## Q3. Third highest salary  ·  🟡 medium

Show the **3rd highest different** salary in the company as `third_highest`.

*Expected result: 1 row · columns: `third_highest`*

**How it works:** DENSE_RANK over all salaries (no PARTITION BY); rank 3 = the third different salary. DISTINCT shows it once.

In [ ]:
%%oracle
SELECT DISTINCT salary AS third_highest
FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS drnk
      FROM employees) t
WHERE drnk = 3

,third_highest
0,140000.0


**Why / traps:** DENSE_RANK + DISTINCT, because several people can share the 3rd salary. Other ways: `OFFSET 2` on distinct salaries; or a correlated COUNT(DISTINCT).

## Q4. Running total of revenue  ·  🟡 medium

For **completed** orders in 2024 (each order once), show `order_date`, `daily_revenue` and `running_total` (sum of all days so far in 2024).

*Expected result: 25 rows · columns: `order_date`, `daily_revenue`, `running_total`*

**How it works:** 1) One row per day (SUM per date). 2) `SUM(...) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING → CURRENT ROW)` adds all days so far.

In [ ]:
%%oracle
WITH daily AS (
    SELECT order_date, SUM(amount) AS daily_revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
      AND order_date >= DATE '2024-01-01' AND order_date < DATE '2025-01-01'
    GROUP BY order_date
)
SELECT order_date, daily_revenue,
       SUM(daily_revenue) OVER (ORDER BY order_date
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily
ORDER BY order_date

,order_date,daily_revenue,running_total
0,2024-01-08,82.0,82
1,2024-01-13,45.0,127
2,2024-01-21,80.0,207
3,2024-02-06,140.0,347
4,2024-02-07,62.0,409
5,2024-02-21,312.0,721
6,2024-03-05,306.0,1027
7,2024-03-13,40.0,1067
8,2024-03-22,30.0,1097
9,2024-04-10,165.0,1262


**Why / traps:** Aggregate to one row per day **first**, then run the window. A day with only NULL amounts gives NULL daily revenue; SUM over the window skips it.

## Q5. Salary vs department average  ·  🟢 easy

Show `name`, `dept_id`, `salary`, `dept_avg` (2 decimals) and `diff` = salary − dept_avg (2 decimals) for employees with a department.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `dept_avg`, `diff`*

**How it works:** `AVG(salary) OVER (PARTITION BY dept_id)` puts the department average on every row; subtract it from the salary.

In [ ]:
%%oracle
SELECT name, dept_id, salary,
       ROUND(AVG(salary) OVER (PARTITION BY dept_id), 2)          AS dept_avg,
       ROUND(salary - AVG(salary) OVER (PARTITION BY dept_id), 2) AS diff
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,dept_avg,diff
0,Alice Johnson,1,250000.0,134285.71,115714.29
1,Bob Smith,1,150000.0,134285.71,15714.29
2,Carol White,1,120000.0,134285.71,-14285.71
3,David Brown,1,120000.0,134285.71,-14285.71
4,Eva Green,1,110000.0,134285.71,-24285.71
5,Frank Moore,1,95000.0,134285.71,-39285.71
6,Grace Lee,1,95000.0,134285.71,-39285.71
7,Henry Clark,2,140000.0,90833.33,49166.67
8,Irene Lewis,2,90000.0,90833.33,-833.33
9,Jack Walker,2,90000.0,90833.33,-833.33


**Why / traps:** With PARTITION BY and **no** ORDER BY, the window is the whole group: every row gets the group total/average.

## Q6. Share of department payroll  ·  🟢 easy

Show `name`, `dept_id`, `salary` and `pct_of_dept` = salary as % of the department's total salary, rounded to 1 decimal. Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `pct_of_dept`*

**How it works:** `SUM(salary) OVER (PARTITION BY dept_id)` = department total on each row; salary / total × 100 = share.

In [ ]:
%%oracle
SELECT name, dept_id, salary,
       ROUND(100 * salary / SUM(salary) OVER (PARTITION BY dept_id), 1) AS pct_of_dept
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,pct_of_dept
0,Alice Johnson,1,250000.0,26.6
1,Bob Smith,1,150000.0,16.0
2,Carol White,1,120000.0,12.8
3,David Brown,1,120000.0,12.8
4,Eva Green,1,110000.0,11.7
5,Frank Moore,1,95000.0,10.1
6,Grace Lee,1,95000.0,10.1
7,Henry Clark,2,140000.0,25.7
8,Irene Lewis,2,90000.0,16.5
9,Jack Walker,2,90000.0,16.5


## Q7. Days since the previous order (LAG)  ·  🟡 medium

For each customer's orders (each order once), show `customer_id`, `order_id`, `order_date`, `prev_order_date` and `days_since_prev` (whole days). The first order of a customer has NULLs.

*Expected result: 104 rows · columns: `customer_id`, `order_id`, `order_date`, `prev_order_date`, `days_since_prev`*

**How it works:** `LAG(order_date)` over each customer's orders (sorted by date) gives the previous order date; subtract the two dates for the gap.

In [ ]:
%%oracle
SELECT customer_id, order_id, order_date,
       LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_order_date,
       order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id)
           AS days_since_prev
FROM (SELECT DISTINCT order_id, customer_id, order_date FROM orders)

,customer_id,order_id,order_date,prev_order_date,days_since_prev
0,1,1026,2024-05-10,NaT,NULL
1,2,1104,2025-12-28,NaT,NULL
2,3,1003,2023-06-25,NaT,NULL
3,4,1065,2025-05-04,NaT,NULL
4,5,1057,2025-03-21,NaT,NULL
5,6,1034,2024-08-04,NaT,NULL
6,7,1002,2023-06-13,NaT,NULL
7,8,1022,2024-03-31,NaT,NULL
8,9,1056,2025-03-03,NaT,NULL
9,9,1071,2025-06-16,2025-03-03,105.0


**Why / traps:** `LAG(col, n, default)` = value n rows **before**. `LEAD` = n rows **after**.

Date difference: `later - earlier` (a number of days). Oracle DATE has a time part; here it is always 00:00.

## Q8. Month-over-month change  ·  🔴 hard

Monthly revenue of completed orders (each order once). Show `month` (first day of the month), `revenue`, `prev_revenue`, and `pct_change` = (revenue − prev) / prev × 100, 1 decimal.

*Expected result: 27 rows · columns: `month`, `revenue`, `prev_revenue`, `pct_change`*

**How it works:** 1) CTE: revenue per month. 2) `LAG(revenue)` = last month's value. 3) (this − last) / last × 100.

In [ ]:
%%oracle
WITH monthly AS (
    SELECT TRUNC(order_date, 'MM') AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
    GROUP BY TRUNC(order_date, 'MM')
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
       ROUND(100 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month

,month,revenue,prev_revenue,pct_change
0,2023-03-01,135,NULL,NULL
1,2023-06-01,115,135.0,-14.8
2,2023-08-01,10,115.0,-91.3
3,2023-11-01,155,10.0,1450.0
4,2023-12-01,102,155.0,-34.2
5,2024-01-01,207,102.0,102.9
6,2024-02-01,514,207.0,148.3
7,2024-03-01,376,514.0,-26.8
8,2024-04-01,165,376.0,-56.1
9,2024-05-01,210,165.0,27.3


**Why / traps:** Months with no completed orders are **missing**, so LAG compares with an older month. To be exact, build a full month calendar first (topic 08). Division by zero: BigQuery has `SAFE_DIVIDE`; elsewhere use `NULLIF(prev, 0)`.

First day of month: `TRUNC(d, 'MM')`.

## Q9. 3-month moving average  ·  🟡 medium

Same monthly revenue. Show `month`, `revenue`, and `moving_avg_3` = average of this month and the 2 months before it (by row), 2 decimals.

*Expected result: 27 rows · columns: `month`, `revenue`, `moving_avg_3`*

**How it works:** Average over a frame of 3 rows: `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = this month and the 2 rows before.

In [ ]:
%%oracle
WITH monthly AS (
    SELECT TRUNC(order_date, 'MM') AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
    GROUP BY TRUNC(order_date, 'MM')
)
SELECT month, revenue,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM monthly
ORDER BY month

,month,revenue,moving_avg_3
0,2023-03-01,135,135.00
1,2023-06-01,115,125.00
2,2023-08-01,10,86.67
3,2023-11-01,155,93.33
4,2023-12-01,102,89.00
5,2024-01-01,207,154.67
6,2024-02-01,514,274.33
7,2024-03-01,376,365.67
8,2024-04-01,165,351.67
9,2024-05-01,210,250.33


**Why / traps:** `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = 3 rows. The first 2 months average fewer rows.

## Q10. Salary quartiles (NTILE)  ·  🟢 easy

Split all employees into 4 groups by salary (highest first): `name`, `salary`, `quartile` (1–4). Break ties by `emp_id`.

*Expected result: 30 rows · columns: `name`, `salary`, `quartile`*

**How it works:** `NTILE(4)` sorts the rows and cuts them into 4 groups of (almost) the same size.

In [ ]:
%%oracle
SELECT name, salary,
       NTILE(4) OVER (ORDER BY salary DESC, emp_id) AS quartile
FROM employees

,name,salary,quartile
0,Alice Johnson,250000.0,1
1,Bob Smith,150000.0,1
2,Henry Clark,140000.0,1
3,Rachel Nelson,135000.0,1
4,Nathan King,130000.0,1
5,Carol White,120000.0,1
6,David Brown,120000.0,1
7,Yara Campbell,115000.0,1
8,Eva Green,110000.0,2
9,Victor Roberts,105000.0,2


**Why / traps:** 30 rows / 4 groups: the first 2 groups get 8 rows, the last 2 get 7.

## Q11. Top earner on every row (FIRST_VALUE)  ·  🟡 medium

Show `name`, `dept_id`, `salary` and `top_earner` = the name of the highest-paid person in the same department (ties: first by name). Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `top_earner`*

**How it works:** `FIRST_VALUE(name)` over each department sorted by salary high first returns the top earner's name on every row.

In [ ]:
%%oracle
SELECT name, dept_id, salary,
       FIRST_VALUE(name) OVER (PARTITION BY dept_id ORDER BY salary DESC, name) AS top_earner
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,top_earner
0,Alice Johnson,1,250000.0,Alice Johnson
1,Bob Smith,1,150000.0,Alice Johnson
2,Carol White,1,120000.0,Alice Johnson
3,David Brown,1,120000.0,Alice Johnson
4,Eva Green,1,110000.0,Alice Johnson
5,Frank Moore,1,95000.0,Alice Johnson
6,Grace Lee,1,95000.0,Alice Johnson
7,Henry Clark,2,140000.0,Henry Clark
8,Irene Lewis,2,90000.0,Henry Clark
9,Jack Walker,2,90000.0,Henry Clark


**Why / traps:** Trap: `LAST_VALUE(...) OVER (ORDER BY ...)` gives the **current** row, because the default frame ends at the current row. Add `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

## Q12. Latest order per customer  ·  🟡 medium

For each customer, show their latest order: `customer_id`, `order_id`, `order_date`, `amount`. If two orders are on the same day, take the bigger `order_id`. One row per customer.

*Expected result: 32 rows · columns: `customer_id`, `order_id`, `order_date`, `amount`*

**How it works:** ROW_NUMBER per customer, newest first (`order_date DESC, order_id DESC`), then keep `rn = 1` = exactly one latest row per customer.

In [ ]:
%%oracle
SELECT customer_id, order_id, order_date, amount
FROM (SELECT o.*,
             ROW_NUMBER() OVER (PARTITION BY customer_id
                                ORDER BY order_date DESC, order_id DESC) AS rn
      FROM orders o) t
WHERE rn = 1

,customer_id,order_id,order_date,amount
0,1,1026,2024-05-10,40.0
1,2,1104,2025-12-28,135.0
2,3,1003,2023-06-25,70.0
3,4,1065,2025-05-04,40.0
4,5,1057,2025-03-21,90.0
5,6,1034,2024-08-04,42.0
6,7,1002,2023-06-13,45.0
7,8,1022,2024-03-31,225.0
8,9,1071,2025-06-16,12.0
9,10,1093,2025-09-28,69.0


**Why / traps:** ROW_NUMBER = exactly one row per group. It also removes the duplicate rows for free. Always add a tie-breaker to ORDER BY, so the result is the same every run.

## Q13. ROWS vs RANGE (ties in a running total)  ·  🔴 hard

Department 2 employees ordered by salary (low → high). Show `name`, `salary`, `rows_total` (frame ROWS ... CURRENT ROW, ties broken by emp_id) and `range_total` (frame RANGE ... CURRENT ROW). Why are they different for the 90,000 tie?

*Expected result: 6 rows · columns: `name`, `salary`, `rows_total`, `range_total`*

**How it works:** ROWS adds one physical row at a time. RANGE adds **all rows with the same salary** at once, so tied rows show the same total.

In [ ]:
%%oracle
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary, emp_id ROWS  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
       SUM(salary) OVER (ORDER BY salary         RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total
FROM employees
WHERE dept_id = 2
ORDER BY salary, emp_id

,name,salary,rows_total,range_total
0,Mia Young,65000.0,65000,65000
1,Leo Allen,70000.0,135000,135000
2,Irene Lewis,90000.0,225000,405000
3,Jack Walker,90000.0,315000,405000
4,Karen Hall,90000.0,405000,405000
5,Henry Clark,140000.0,545000,545000


**Why / traps:** ROWS counts physical rows. RANGE includes **all rows with the same ORDER BY value** (peers). The default frame with ORDER BY is RANGE: that is why running totals jump on ties.